# 02 — Feature Engineering
## dataplace.ai — Zadanie rekrutacyjne Data Scientist 2026

Celem tego notebooka jest obliczenie cech przestrzennych dla każdej z 50 lokalizacji sklepów klienta.
Cechy będą podstawą modelu ML szacującego potencjał sprzedażowy lokalizacji.

### Źródła danych
| Źródło | Cechy |
|---|---|
| Snowflake — RECRUITMENT_TRACES | Ruch pieszy: `signal_count`, `unique_users`, `peak_morning_signals` |
| AWS S3 — recruitment_buildings | Zabudowa: `building_count`, `residential_count`, `commercial_count`, `building_density` |
| AWS S3 — recruitment_population | Demografia: `pop_total`, `pop_households` |
| ds_poi.csv | POI: `poi_restaurant`, `poi_school`, `poi_bus_stop` ... |
| ds_competitors.csv | Konkurencja: `competitor_count`, `dist_nearest_competitor_m` |

### Metodologia
Dla każdej lokalizacji tworzymy bufor **500m** i obliczamy agregaty przestrzenne
w jego obrębie. Wynikiem jest tabela `features.csv` z 50 wierszami i 23 cechami.

In [1]:
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely import wkt
import snowflake.connector
import os

load_dotenv()

DATA_DIR      = r"C:\Users\slast\OneDrive\Pulpit\geo_zadanie\01_data\raw"
PROCESSED_DIR = r"C:\Users\slast\OneDrive\Pulpit\geo_zadanie\01_data\procced"

BUFFER_RADIUS_M = 500

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120

print("✅ Biblioteki załadowane")

✅ Biblioteki załadowane


In [2]:
# Wczytanie obszaru analizy i wyznaczenie bbox
df_area   = pd.read_csv(os.path.join(DATA_DIR, 'ds_analysis_area.csv'))
area_geom = wkt.loads(df_area['geometry'].iloc[0])
minx, miny, maxx, maxy = area_geom.bounds

BBOX = {
    'lat_min': miny,
    'lat_max': maxy,
    'lng_min': minx,
    'lng_max': maxx
}

print(f"✅ BBOX wyznaczony dynamicznie z ds_analysis_area.csv")
print(f"   lat: {BBOX['lat_min']:.4f} — {BBOX['lat_max']:.4f}")
print(f"   lng: {BBOX['lng_min']:.4f} — {BBOX['lng_max']:.4f}")

✅ BBOX wyznaczony dynamicznie z ds_analysis_area.csv
   lat: 49.9600 — 50.4800
   lng: 18.5400 — 20.1200


In [3]:
# Wczytanie lokalnych plików CSV
df_locations   = pd.read_csv(os.path.join(DATA_DIR, 'ds_locations.csv'))
df_competitors = pd.read_csv(os.path.join(DATA_DIR, 'ds_competitors.csv'))
df_poi         = pd.read_csv(os.path.join(DATA_DIR, 'ds_poi.csv'))

# AWS (lokalnie)
df_buildings  = pd.read_csv(os.path.join(DATA_DIR, 'recruitment_buildings.csv.gz'),  compression='gzip')
df_population = pd.read_csv(os.path.join(DATA_DIR, 'recruitment_population.csv.gz'), compression='gzip')

print(f"✅ Lokalizacje:  {len(df_locations)}")
print(f"✅ Konkurencja:  {len(df_competitors)}")
print(f"✅ POI:          {len(df_poi)}")
print(f"✅ Budynki:      {len(df_buildings):,}")
print(f"✅ Populacja:    {len(df_population):,}")

✅ Lokalizacje:  50
✅ Konkurencja:  98
✅ POI:          335
✅ Budynki:      955,612
✅ Populacja:    450,140


In [4]:
# Lokalizacje klienta
gdf_locations = gpd.GeoDataFrame(
    df_locations,
    geometry=gpd.points_from_xy(df_locations['lng'], df_locations['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

# Konkurencja
gdf_competitors = gpd.GeoDataFrame(
    df_competitors,
    geometry=gpd.points_from_xy(df_competitors['lng'], df_competitors['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

# POI
gdf_poi = gpd.GeoDataFrame(
    df_poi,
    geometry=gpd.points_from_xy(df_poi['lng'], df_poi['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

# Populacja (filtrujemy do bbox)
mask = (
    df_population['lat'].between(BBOX['lat_min'], BBOX['lat_max']) &
    df_population['lng'].between(BBOX['lng_min'], BBOX['lng_max'])
)
gdf_population = gpd.GeoDataFrame(
    df_population[mask].copy(),
    geometry=gpd.points_from_xy(df_population[mask]['lng'], df_population[mask]['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

# Budynki
gdf_buildings = gpd.GeoDataFrame(
    df_buildings,
    geometry=df_buildings['geometry'].apply(wkt.loads),
    crs='EPSG:4326'
).to_crs('EPSG:2180')

print(f"✅ gdf_locations:   {len(gdf_locations)} punktów")
print(f"✅ gdf_competitors: {len(gdf_competitors)} punktów")
print(f"✅ gdf_poi:         {len(gdf_poi)} punktów")
print(f"✅ gdf_population:  {len(gdf_population):,} punktów")
print(f"✅ gdf_buildings:   {len(gdf_buildings):,} budynków")
print(f"✅ CRS: {gdf_locations.crs}")

✅ gdf_locations:   50 punktów
✅ gdf_competitors: 98 punktów
✅ gdf_poi:         335 punktów
✅ gdf_population:  450,140 punktów
✅ gdf_buildings:   955,612 budynków
✅ CRS: EPSG:2180


In [5]:
gdf_buffers = gdf_locations[['location_id', 'geometry']].copy()
gdf_buffers['geometry'] = gdf_locations.geometry.buffer(BUFFER_RADIUS_M)

print(f"✅ Bufory {BUFFER_RADIUS_M}m utworzone dla {len(gdf_buffers)} lokalizacji")

✅ Bufory 500m utworzone dla 50 lokalizacji


## Feature Engineering

Dla każdej lokalizacji obliczamy **23 cechy** w buforze 500m, pogrupowane w 5 kategorii:

| Grupa | Liczba cech | Źródło |
|---|---|---|
| POI | 9 | ds_poi.csv |
| Konkurencja | 2 | ds_competitors.csv |
| Demografia | 2 | recruitment_population.csv.gz |
| Zabudowa | 6 | recruitment_buildings.csv.gz |
| Foot traffic | 3 | Snowflake |

In [6]:
# Spatial join — POI w buforze 1km (500m za mało — POI zbyt rzadkie)
gdf_buffers_1km = gdf_locations[['location_id', 'geometry']].copy()
gdf_buffers_1km['geometry'] = gdf_locations.geometry.buffer(1000)

joined_poi = gpd.sjoin(
    gdf_poi,
    gdf_buffers_1km,
    how='inner',
    predicate='within'
)

poi_pivot = joined_poi.groupby(['location_id', 'category']).size().unstack(fill_value=0)
poi_pivot.columns = [f'poi_{c}' for c in poi_pivot.columns]
poi_pivot['poi_total'] = poi_pivot.sum(axis=1)
poi_pivot = poi_pivot.reset_index()

# Uzupełniamy brakujące lokalizacje i kolumny zerami
poi_features = df_locations[['location_id']].merge(poi_pivot, on='location_id', how='left').fillna(0)

for cat in ['poi_bank', 'poi_bus_stop', 'poi_hospital', 'poi_mall',
            'poi_park', 'poi_pharmacy', 'poi_restaurant', 'poi_school']:
    if cat not in poi_features.columns:
        poi_features[cat] = 0

poi_features['poi_total'] = poi_features[
    [c for c in poi_features.columns if c.startswith('poi_') and c != 'poi_total']
].sum(axis=1)

print(f"✅ POI features: {poi_features.shape}")
print(f"Lokalizacji z POI w buforze 1km: {(poi_features['poi_total'] > 0).sum()} z 50")

✅ POI features: (50, 10)
Lokalizacji z POI w buforze 1km: 24 z 50


In [7]:
# Liczba konkurentów w buforze 500m
joined_comp = gpd.sjoin(
    gdf_competitors,
    gdf_buffers,
    how='inner',
    predicate='within'
)

comp_count = joined_comp.groupby('location_id').size().rename('competitor_count').reset_index()

# Odległość do najbliższego konkurenta
dist_nearest = []
for _, loc in gdf_locations.iterrows():
    d = gdf_competitors.geometry.distance(loc.geometry).min()
    dist_nearest.append({
        'location_id': loc['location_id'],
        'dist_nearest_competitor_m': round(d, 1)
    })

df_dist = pd.DataFrame(dist_nearest)

# Złączenie
comp_features = df_locations[['location_id']].merge(comp_count, on='location_id', how='left')
comp_features = comp_features.merge(df_dist, on='location_id', how='left')
comp_features['competitor_count'] = comp_features['competitor_count'].fillna(0)

print(f"✅ Competitor features: {comp_features.shape}")
display(comp_features.describe().round(1))

✅ Competitor features: (50, 3)


,competitor_count,dist_nearest_competitor_m
count,50.0,50.0
mean,0.1,2541.4
std,0.3,2867.4
min,0.0,42.2
25%,0.0,1047.7
50%,0.0,1711.2
75%,0.0,2892.2
max,1.0,13842.3


In [8]:
joined_pop = gpd.sjoin(
    gdf_population,
    gdf_buffers,
    how='inner',
    predicate='within'
)

demo_features = joined_pop.groupby('location_id').agg(
    pop_total      = ('total', 'sum'),
    pop_households = ('households', 'sum'),
    avg_hh_size    = ('household_size', 'mean')
).reset_index()

demo_features = df_locations[['location_id']].merge(demo_features, on='location_id', how='left').fillna(0)
demo_features['avg_hh_size'] = demo_features['avg_hh_size'].round(2)

print(f"✅ Demo features: {demo_features.shape}")
display(demo_features.describe().round(1))

✅ Demo features: (50, 4)


,pop_total,pop_households,avg_hh_size
count,50.0,50.0,50.0
mean,3821.4,1497.7,3.5
std,2661.3,1113.6,1.4
min,117.0,31.0,1.9
25%,1349.5,448.2,2.4
50%,3550.5,1523.5,3.2
75%,6221.8,2162.8,3.9
max,8526.0,3908.0,8.0


In [9]:
joined_buildings = gpd.sjoin(
    gdf_buildings,
    gdf_buffers,
    how='inner',
    predicate='intersects'
)

# Grupowanie typów budynków
residential = ['budynkiMieszkalneJednorodzinne', 'budynkiODwochMieszkaniach',
               'budynkiOTrzechIWiecejMieszkaniach', 'budynkiZbiorowegoZamieszkania']
commercial  = ['budynkiHandlowoUslugowe', 'budynkiBiurowe', 'budynkiPrzemyslowe']

joined_buildings['is_residential'] = joined_buildings['funogolnabudynku_desc'].isin(residential)
joined_buildings['is_commercial']  = joined_buildings['funogolnabudynku_desc'].isin(commercial)

building_features = joined_buildings.groupby('location_id').agg(
    building_count      = ('area', 'count'),
    total_building_area = ('area', 'sum'),
    avg_building_area   = ('area', 'mean'),
    residential_count   = ('is_residential', 'sum'),
    commercial_count    = ('is_commercial', 'sum'),
).reset_index()

building_features['building_density'] = (
    building_features['building_count'] / (3.14159 * (BUFFER_RADIUS_M/1000)**2)
).round(1)

building_features = df_locations[['location_id']].merge(
    building_features, on='location_id', how='left').fillna(0)

print(f"✅ Building features: {building_features.shape}")
display(building_features.describe().round(1))

✅ Building features: (50, 7)


,building_count,total_building_area,avg_building_area,residential_count,commercial_count,building_density
count,50.0,50.0,50.0,50.0,50.0,50.0
mean,443.5,103395.0,255.1,256.2,49.7,564.7
std,242.6,57182.7,117.9,139.7,35.1,308.9
min,54.0,12620.3,101.0,35.0,8.0,68.8
25%,276.0,63461.9,182.2,148.8,23.0,351.4
50%,406.0,92518.0,223.6,255.5,39.0,517.0
75%,548.5,133579.5,311.4,333.0,66.0,698.3
max,1236.0,304561.9,610.9,707.0,131.0,1573.7


In [10]:
conn = snowflake.connector.connect(
    account=os.getenv('SF_ACCOUNT'),
    user=os.getenv('SF_USER'),
    password=os.getenv('SF_PASSWORD'),
    warehouse=os.getenv('SF_WAREHOUSE'),
    database=os.getenv('SF_DATABASE'),
    schema=os.getenv('SF_SCHEMA'),
    role=os.getenv('SF_ROLE')
)
cur = conn.cursor()
print("✅ Połączono ze Snowflake")

✅ Połączono ze Snowflake


In [11]:
foot_traffic_records = []

print(f"Pobieranie foot traffic dla {len(df_locations)} lokalizacji...")

for i, row in df_locations.iterrows():
    lat, lng, loc_id = row['lat'], row['lng'], row['location_id']

    cur.execute(f"""
        SELECT 
            '{loc_id}'                                      AS location_id,
            COUNT(*)                                        AS signal_count,
            COUNT(DISTINCT proxi_user_id)                   AS unique_users,
            SUM(CASE WHEN HOUR(TRY_TO_TIMESTAMP(occured_at))
                BETWEEN 6 AND 9 THEN 1 ELSE 0 END)         AS peak_morning_signals
        FROM RECRUITMENT_TRACES
        WHERE TRY_CAST(latitude  AS FLOAT) BETWEEN {lat - 0.005} AND {lat + 0.005}
          AND TRY_CAST(longitude AS FLOAT) BETWEEN {lng - 0.005} AND {lng + 0.005}
          AND TRY_TO_TIMESTAMP(occured_at) BETWEEN '2020-07-01' AND '2020-07-08'
          AND ST_DISTANCE(
                ST_MAKEPOINT(TRY_CAST(longitude AS FLOAT), TRY_CAST(latitude AS FLOAT)),
                ST_MAKEPOINT({lng}, {lat})
              ) <= {BUFFER_RADIUS_M}
    """)

    foot_traffic_records.append(cur.fetchone())

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/50 gotowe...")

df_foot_traffic = pd.DataFrame(
    foot_traffic_records,
    columns=['location_id', 'signal_count', 'unique_users', 'peak_morning_signals']
)

print(f"\n✅ Foot traffic: {df_foot_traffic.shape}")
display(df_foot_traffic.describe().round(1))

Pobieranie foot traffic dla 50 lokalizacji...
  10/50 gotowe...
  20/50 gotowe...
  30/50 gotowe...
  40/50 gotowe...
  50/50 gotowe...

✅ Foot traffic: (50, 4)


,signal_count,unique_users,peak_morning_signals
count,50.0,50.0,50.0
mean,12125.5,239.2,2834.1
std,9940.1,161.4,2496.8
min,441.0,24.0,90.0
25%,2913.8,104.0,709.8
50%,10887.0,214.0,2456.0
75%,17879.0,315.2,4193.0
max,47250.0,782.0,13000.0


In [12]:
df_features = df_locations[['location_id', 'lat', 'lng', 'monthly_revenue']].copy()

for df_part, name in [
    (df_foot_traffic,   'foot_traffic'),
    (building_features, 'buildings'),
    (demo_features,     'demographics'),
    (poi_features,      'poi'),
    (comp_features,     'competitors'),
]:
    df_features = df_features.merge(df_part, on='location_id', how='left')
    print(f"  + {name}: {df_part.shape[1]-1} cech")

df_features = df_features.fillna(0)

print(f"\n✅ Finalny dataset: {df_features.shape[0]} lokalizacji × {df_features.shape[1]} kolumn")
display(df_features.head(3))

  + foot_traffic: 3 cech
  + buildings: 6 cech
  + demographics: 3 cech
  + poi: 9 cech
  + competitors: 2 cech

✅ Finalny dataset: 50 lokalizacji × 27 kolumn


,location_id,lat,lng,monthly_revenue,signal_count,unique_users,peak_morning_signals,building_count,total_building_area,avg_building_area,...,poi_bus_stop,poi_hospital,poi_mall,poi_park,poi_pharmacy,poi_restaurant,poi_school,poi_total,competitor_count,dist_nearest_competitor_m
0,LOC_001,50.226272,19.072973,245633.74,16052,216,3644,370,62829.45,169.809324,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,262.8
1,LOC_002,50.299999,18.674625,333393.33,21249,521,5691,442,180292.51,407.901606,...,0.0,0.0,0.0,0.0,1.0,2.0,2.0,5.0,0.0,2567.5
2,LOC_003,50.042894,19.968676,296109.69,4547,241,1248,345,126428.17,366.458464,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,619.2


In [13]:
df_features.to_csv(os.path.join(PROCESSED_DIR, 'features.csv'), index=False)

conn.close()

print("✅ features.csv zapisany")
print("✅ Połączenie Snowflake zamknięte")

✅ features.csv zapisany
✅ Połączenie Snowflake zamknięte


## Podsumowanie Feature Engineering

Finalny dataset: **50 lokalizacji × 23 cechy** (+ location_id, lat, lng, monthly_revenue)

| Grupa | Cechy | Promień |
|---|---|---|
| Foot traffic | signal_count, unique_users, peak_morning_signals | 500m |
| Zabudowa | building_count, total_building_area, avg_building_area, residential_count, commercial_count, building_density | 500m |
| Demografia | pop_total, pop_households, avg_hh_size | 500m |
| POI | poi_bank, poi_bus_stop, poi_hospital, poi_mall, poi_park, poi_pharmacy, poi_restaurant, poi_school, poi_total | 1000m |
| Konkurencja | competitor_count, dist_nearest_competitor_m | 500m / bez limitu |

Dataset zapisany do `01_data/procced/features.csv`.
Gotowy do modelowania w notebooku `03_model.ipynb`.